Создайте программу, которая распознает и подсчитывает только автомобили и автобусы (2 и 5 класс) с видеофайла.

In [ ]:
import cv2
from ultralytics import YOLO

# Загрузка модели YOLO
model = YOLO('yolov8n.pt')

# Загрузка видео файла
video_path = 'zadanie.mp4'  # Укажите путь к вашему видео
cap = cv2.VideoCapture(video_path)

while True:
    # Чтение кадра
    ret, frame = cap.read()
    if not ret:
        print("Видео закончилось или не удалось прочитать кадр")
        break
    frame = cv2.resize(frame, (640, 480))
    # Детекция объектов с YOLO
    results = model(frame)     
    # Проверяем, есть ли результаты
    
    if len(results) > 0:
        result = results[0]
        # Обрабатываем каждый обнаруженный объект
        for box in result.boxes:
            class_id = int(box.cls[0])
            confidence = float(box.conf[0])
            # Проверяем, является ли объект автомобилем или автобусом

                # Получаем координаты bounding box

                
                # Рисуем bounding box

    # Отображение результата
    cv2.imshow('YOLO Video Detection', frame)

    # Выход по нажатию 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Создайте программу, которая выполняет сегментацию и подсчитывает только автомобили и автобусы (классы 2 и 5) с видеофайла, используя точные маски вместо bounding boxes.

In [ ]:
import cv2
from ultralytics import YOLO
import supervision as sv
import numpy as np

# Загрузка модели YOLO для сегментации
model = YOLO('yolov8n-seg.pt')

# Загрузка видео файла
video_path = 'zadanie.mp4'
cap = cv2.VideoCapture(video_path)

# Создаем аннотаторы для сегментации
mask_annotator = sv.MaskAnnotator()
label_annotator = sv.LabelAnnotator()
box_annotator = sv.BoxAnnotator()

# Классы для детекции: car=2, bus=5
target_classes = [2, 5]
class_names = {2: 'CAR', 5: 'BUS'}
colors = {2: (0,255,0),  # Зеленый для автомобилей
          5: (255,0,0)}  # Красный для автобусов

while True:
    ret, frame = cap.read()
    if not ret:
        print("Видео закончилось")
        break
    
    frame = cv2.resize(frame, (640, 480))
    
    # Сегментация объектов с YOLO

    
    # Конвертируем результаты в формат supervision

    
    # Фильтруем только автомобили и автобусы
    vehicle_indices = [
        i for i, class_id in enumerate(detections.class_id) 
        if class_id in target_classes
    ]
    
    vehicle_detections = detections[vehicle_indices]
    
    # Создаем кастомные цвета для аннотатора
    class_color_map = {}
    for class_id in target_classes:
        class_color_map[class_id] = colors[class_id]

    # Создаем подписи
    labels = [
        f"{class_names[class_id]}"
        for class_id in (vehicle_detections.class_id)
    ]
    
    # Аннотируем
    
    # Отображение результата
    cv2.imshow('Vehicle Segmentation with Supervision', annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# Контроль безопасности робота на производственной линии

Создайте систему для отслеживания и мониторинга промышленных роботов на производственной линии.

# Цель:
- Обнаруживать людей в рабочей зоне
- Сигнализировать о нарушениях безопасных расстояний

# Основные функции:
1. Найти всех людей в кадре
2. Дать ID каждому человеку
3. Определять зоны безопасности вокруг роботов (останавливать робота при появлении человека в кадре)
4. Показать статистику безопасности (количество людей)

# Требования:

- Обводить людей прямоугольниками
- Показывать ID каждого работника
- Сигнализировать при нарушении безопасного расстояния

# Входные данные:
- Видео с камеры
- Модель YOLO (yolov8n.pt)

In [ ]:
import cv2
from ultralytics import YOLO
import supervision as sv
import numpy as np

# Загрузка модели
model = YOLO('yolov8n.pt')
tracker = sv.ByteTrack()

# Классы для поиска
person_class = 0

video_path = 'Zadanie.mov'
cap = cv2.VideoCapture(0)

# Создаем аннотаторы для сегментации
mask_annotator = sv.MaskAnnotator()
label_annotator = sv.LabelAnnotator()
box_annotator = sv.BoxAnnotator()

while True:
    ret, frame = cap.read()
    if not ret:
        print("Видео закончилось")
        break
    
    frame = cv2.resize(frame, (640, 480))
    
    # Сегментация объектов с YOLO
    results = model(frame)
    
    # Конвертируем результаты в формат supervision
    detections = sv.Detections.from_ultralytics(results[0])
    # Трекинг объектов
    detections = tracker.update_with_detections(detections)
    
    # Создаем подписи с ID
    labels = [
        f"ID:{tracker_id} {model.names[class_id]}"
        for class_id, tracker_id in zip(detections.class_id, detections.tracker_id)
    ]
    
    # Аннотируем
    annotated_frame = mask_annotator.annotate(frame.copy(), detections)
    annotated_frame = box_annotator.annotate(annotated_frame, detections)
    annotated_frame = label_annotator.annotate(annotated_frame, detections, labels)
    
    # Отображение результата
    cv2.imshow('Segmentation with Tracking', annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()